In [ ]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'

    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')

    os.chdir(repo_path + '/data_pipeline')
    print("Installing base dependencies...")
    os.system('pip install -q pandas scikit-learn transformers torch evaluate huggingface_hub')

    if not os.path.exists("models/benchmark/ConLID/repo"):
        print("Setting up ConLID dependencies...")
        os.makedirs("models/benchmark/ConLID", exist_ok=True)
        os.system('git clone https://github.com/epfl-nlp/language-identification.git models/benchmark/ConLID/repo')
        os.system('pip install -q -r models/benchmark/ConLID/repo/requirements.txt')

    print("Setup complete!")


Running in Google Colab. Setting up environment...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Installing base dependencies...
Setup complete!


In [ ]:
# NOTE: Ensure you have `torch`, `evaluate`, and the ConLID repo set up locally (`make setup-conlid`).
TRAIN_PATH = "/content/drive/MyDrive/SSLI/processed/public_shared/train.csv"
VAL_PATH = "/content/drive/MyDrive/SSLI/processed/public_shared/val.csv"
output_model_dir = 'models/finetuned/conlid'
batch_size = 8
accumulation_steps = 4
learning_rate = 1e-4
num_epochs = 10


In [ ]:
import pandas as pd

df = pd.read_csv(TRAIN_PATH)

print(df.shape)
print(df.columns)
print(df["label"].value_counts())
df.head()

(60285, 6)
Index(['id', 'text', 'label', 'source', 'subcorpus', 'group_id'], dtype='object')
label
sinhala     26675
pali        23490
sanskrit    10120
Name: count, dtype: int64


,id,text,label,source,subcorpus,group_id
0,0,අධිමාසංදීපනි.,pali,SiDiaC-v2,අධිමාස දීපනය,sidiac2_අධිමාස දීපනය
1,1,නමො අද්වයවාදිනො සම්මා සම්බුද්ධස්ස.,pali,SiDiaC-v2,අධිමාස දීපනය,sidiac2_අධිමාස දීපනය
2,2,1 නමාමී බුද්ධං චතුසච්ච බුද්ධං,pali,SiDiaC-v2,අධිමාස දීපනය,sidiac2_අධිමාස දීපනය
3,3,"නමාමි ධම්මං අධිමොක්ඛ ධම්මං,",pali,SiDiaC-v2,අධිමාස දීපනය,sidiac2_අධිමාස දීපනය
4,4,නමාමි සංඝං හතපාප සංඝං,pali,SiDiaC-v2,අධිමාස දීපනය,sidiac2_අධිමාස දීපනය


In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
import pandas as pd
from huggingface_hub import snapshot_download

# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

REPO_DIR = "models/benchmark/ConLID/repo"

if not os.path.exists(REPO_DIR):
    raise RuntimeError(f"{REPO_DIR} not found. Please run 'make setup-conlid' in the pipeline root first.")

sys.path.append(REPO_DIR)
from model import ConLID  # noqa: E402

print("Downloading ConLID checkpoints...")
checkpoint_dir = os.path.join(REPO_DIR, "checkpoints", "conlid")
snapshot_download(repo_id="epfl-nlp/ConLID", local_dir=checkpoint_dir)

print("Loading ConLID model...")
model = ConLID.from_pretrained(dir=checkpoint_dir)
device = model.device

# Target Mapping (ConLID uses FLORES-200 style codes mostly)
TARGET_LANGUAGES = {
    "eng": "eng_Latn",
    "hin": "hin_Deva",
    "arb": "arb_Arab",
    "fra": "fra_Latn",
    "deu": "deu_Latn",
    "ben": "ben_Beng",
    "tam": "tam_Taml",
    "sin": "sin_Sinh",
    "san": "san_Deva",
    "pli": "pli_Latn",  # Assign a sensible code for Pali
}

# Expand Classification Head for missing languages
existing_labels = list(model.id2label.values())
added_labels = []

for short_code, long_code in TARGET_LANGUAGES.items():
    if long_code not in existing_labels:
        idx = len(model.id2label)
        model.id2label[idx] = long_code
        existing_labels.append(long_code)
        added_labels.append(long_code)

if added_labels:
    import numpy as np
    print(f"Expanding classification head to add {len(added_labels)} new labels: {added_labels}")
    old_out_features = model.fc.out_features
    new_out_features = len(model.id2label)

    new_fc = nn.Linear(model.fc.in_features, new_out_features)
    new_fc.weight.data[:old_out_features] = model.fc.weight.data
    new_fc.bias.data[:old_out_features] = model.fc.bias.data

    nn.init.xavier_uniform_(new_fc.weight.data[old_out_features:])
    nn.init.zeros_(new_fc.bias.data[old_out_features:])

    model.fc = new_fc.to(device)
    model.convert_id2label = np.vectorize(model.id2label.get)

print(f"Model ready. Total labels: {len(model.id2label)}")


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading ConLID model...
Expanding classification head to add 1 new labels: ['pli_Latn']
Model ready. Total labels: 2100


In [ ]:
import os

print("Current folder:", os.getcwd())
print(os.listdir())

Current folder: /content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline
['.env.example', '.gitignore', '.python-version', 'Makefile', 'README.md', 'models', 'pyproject.toml', 'scripts', 'uv.lock']


In [ ]:
import glob

matches = glob.glob("/content/**/train.csv", recursive=True)

print(matches)

['/content/drive/MyDrive/SSLI/processed/public_shared/train.csv']


In [ ]:
from torch.utils.data import Dataset, DataLoader

# Reverse lookup for label string to ID
label2id = {v: k for k, v in model.id2label.items()}

MAX_TEXT_CHARS = 2000  # guards against pathologically long outliers blowing up batch memory

class ConLIDDataset(Dataset):
    def __init__(self, jsonl_path, model, label_map):
        self.records = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                rec = json.loads(line)
                mapped_label = label_map.get(rec['label'], rec['label'])
                if mapped_label in label2id:
                    self.records.append((rec['text'], label2id[mapped_label]))

        self.model = model

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        text, label = self.records[idx]
        text = text[:MAX_TEXT_CHARS]
        tokens = self.model._tokenize(text)
        ids = self.model._tokens2ngrams(tokens)
        return torch.tensor(ids, dtype=torch.int), torch.tensor(label, dtype=torch.long)

def collate_fn(batch):
    # Pad sequences to max length in batch
    ids, labels = zip(*batch)
    max_len = max(len(x) for x in ids)
    padded_ids = [torch.cat([x, torch.full((max_len - len(x),), model.pad_id, dtype=torch.int)]) for x in ids]
    return torch.stack(padded_ids), torch.stack(labels)

print("\nLoading Datasets...")
train_dataset = ConLIDDataset(os.path.join(finetune_dir, "train.jsonl"), model, TARGET_LANGUAGES)
val_dataset = ConLIDDataset(os.path.join(finetune_dir, "val_mixed.jsonl"), model, TARGET_LANGUAGES)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")


Streaming output truncated to the last 5000 lines.


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
import evaluate
import numpy as np
from tqdm.auto import tqdm
import copy

metric = evaluate.load("f1")
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

best_f1 = -1
best_model_state = None
patience = 2
patience_counter = 0

print("Starting Fine-tuning...")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, (input_ids, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")):
        input_ids, labels = input_ids.to(device), labels.to(device)


        logits = model(input_ids)
        loss = criterion(logits, labels)
        loss = loss / accumulation_steps

        loss.backward()
        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
            optimizer.step()
            optimizer.zero_grad()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} - Avg Train Loss: {total_loss / len(train_loader):.4f}")

    # Validation
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for input_ids, labels in val_loader:
            input_ids = input_ids.to(device)
            logits = model(input_ids)
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    val_f1 = metric.compute(predictions=all_preds, references=all_labels, average="micro")["f1"]
    print(f"Epoch {epoch+1} - Validation Micro F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered! Restoring best model (F1: {best_f1:.4f})")
            model.load_state_dict(best_model_state)
            break

# Save Model
import safetensors.torch
os.makedirs(output_model_dir, exist_ok=True)
safetensors.torch.save_model(model, os.path.join(output_model_dir, "model.safetensors"))

# Save configs
with open(os.path.join(output_model_dir, "config.json"), 'w') as f:
    config_dict = {
        "vocab_size": model.vocab_size,
        "embedding_size": model.embedding.embedding_dim,
        "num_classes": len(model.id2label),
        "bucket": model.bucket,
        "min_count": model.min_count,
        "minn": model.minn,
        "maxn": model.maxn,
        "aggr": "mean", # standard
        "pad_id": model.pad_id,
        "unk_id": model.unk_id
    }
    json.dump(config_dict, f)

with open(os.path.join(output_model_dir, "vocab.json"), 'w') as f:
    json.dump(model.vocab, f)

with open(os.path.join(output_model_dir, "labels.json"), 'w') as f:
    labels_dict = {v: k for k, v in model.id2label.items()}
    json.dump(labels_dict, f)

print(f"\nFinetuning Complete! Model saved to {output_model_dir}")


## Evaluation: fine-tuned ConLID across ALL benchmark languages

Evaluates the fine-tuned model on every `datasets/preprocessed/*.jsonl` benchmark file, over all target + old languages (not just Sinhala/Pali/Sanskrit), mirroring the "ALL LANGUAGES" reports used for the other finetuned models.

In [ ]:
import glob
from sklearn.metrics import accuracy_score, classification_report, f1_score

benchmark_input_dir = 'datasets/preprocessed'
benchmark_output_dir = 'datasets/benchmark_results'
os.makedirs(benchmark_output_dir, exist_ok=True)

ALL_BENCHMARK_LANGUAGES = [
    "sinhala",
    "pali",
    "sanskrit",
    "sanskrit_deva",
    "english",
    "tamil",
    "hindi",
    "bengali",
    "arabic",
    "french",
    "german",
]

# Short benchmark codes -> readable names used for reporting.
LABEL_MAPPING_ALL = {
    "sin": "sinhala",
    "pli": "pali",
    "eng": "english",
    "tam": "tamil",
    "hin": "hindi",
    "ben": "bengali",
    "arb": "arabic",
    "fra": "french",
    "deu": "german",
}

# Readable name -> FLORES-style code the fine-tuned model was trained to predict.
NAME_TO_MODEL_LABEL = {
    "sinhala": "sin_Sinh",
    "pali": "pli_Latn",
    "sanskrit": "san_Deva",
    "sanskrit_deva": "san_Deva",
    "english": "eng_Latn",
    "tamil": "tam_Taml",
    "hindi": "hin_Deva",
    "bengali": "ben_Beng",
    "arabic": "arb_Arab",
    "french": "fra_Latn",
    "german": "deu_Latn",
}
MODEL_LABEL_TO_NAME = {v: k for k, v in NAME_TO_MODEL_LABEL.items() if k != "sanskrit_deva"}


def map_all_label(row):
    lbl = str(row.get("label", "")).strip()
    src = row.get("source")

    if lbl == "san":
        if src in ["DCS", "SansinNT", "SiDiaC-v2"]:
            return "sanskrit"
        return "sanskrit_deva"

    return LABEL_MAPPING_ALL.get(lbl)


def load_all_benchmark(file_path):
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            mapped = map_all_label(row)
            if mapped:
                row["target_label"] = mapped
                records.append(row)
    return pd.DataFrame(records)


def predict_texts(texts, batch_size=64):
    predictions = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Predicting"):
        chunk = texts[start:start + batch_size]
        labels_batch, _ = model.predict_batched(chunk, k=1)
        for labels in labels_batch:
            code = labels[0] if labels else "unknown"
            predictions.append(MODEL_LABEL_TO_NAME.get(code, code))
    return predictions


benchmark_files = sorted(glob.glob(os.path.join(benchmark_input_dir, "*.jsonl")))
print("Evaluating fine-tuned ConLID across ALL benchmark languages in", benchmark_input_dir, "...")

model.eval()
all_language_summary = []

for file_path in benchmark_files:
    dataset_name = os.path.splitext(os.path.basename(file_path))[0]
    df_all = load_all_benchmark(file_path)

    if df_all.empty:
        print(f"No matching benchmark rows in {dataset_name}.")
        continue

    print(f"\nEvaluating {len(df_all)} samples across ALL benchmark languages from {dataset_name}...")

    results = df_all.copy()
    results["predicted_label"] = predict_texts(results["text"].tolist())

    y_true = results["target_label"]
    y_pred = results["predicted_label"]

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, labels=ALL_BENCHMARK_LANGUAGES, average="macro", zero_division=0)

    print("=" * 65)
    print(f"ALL LANGUAGES BENCHMARK RESULTS (ConLID Finetuned - Evaluated on {dataset_name})")
    print("=" * 65)
    print(f"Accuracy:  {acc*100:.2f}%")
    print(f"Macro F1:  {macro_f1*100:.2f}%")
    print("=" * 65)

    print("\nPer-language breakdown (All Benchmark Languages):\n")
    print(classification_report(y_true, y_pred, labels=ALL_BENCHMARK_LANGUAGES, digits=4, zero_division=0))

    out_csv = os.path.join(benchmark_output_dir, f"conlid_finetuned_all_langs_{dataset_name}.csv")
    results.to_csv(out_csv, index=False)
    print(f"Saved all-languages benchmark predictions to {out_csv}")

    all_language_summary.append({
        "dataset": dataset_name,
        "rows": len(results),
        "accuracy": acc,
        "macro_f1": macro_f1,
    })

all_language_summary_df = pd.DataFrame(all_language_summary)
all_language_summary_df